<a href="https://colab.research.google.com/github/HugoDelgado/AI4MLCysec/blob/main/Module2_AI4CYSEC_Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 Binary Size to Function Count
## A Quadratic Regression Activity with PyGhidra

**Goal**: To use `PyGhidra` to analyze a few binary files and extract features from them such as **function count** to build a quadratic regression model to predict the feature from the **binary file size**.

In this lab we will be using simple binary files from the unix `\bin\` system folder in the Linux system behind the Google Colab environment. We will be doing 5 binaries that we already know, in reality the cybersecurity professional could have a folder full of small binary segments representing the target malware.

In order to show a new model we will use a quadratic $y = a x^2 + b x + c$. This model produces three parameters that represent a parabolic graph. Data is not always linear and we need to use different models, many of them from the basic mathematical functions.

# Install PyGhidra

`PyGhidra` is the official Python module to access **Ghidra**, a reverse engineering tool for cybersecurity. It runs with no graphical user interface which is what we need for our automated dataset creation pipeline.

In [ ]:
import os
# 1. Install Java JDK 25 and download Ghidra 12.1.3
!apt-get update -qq && apt-get install openjdk-25-jdk-headless -y -qq
!wget -q https://github.com/NationalSecurityAgency/ghidra/releases/download/Ghidra_12.1.3_build/ghidra_12.1.3_PUBLIC_20260817.zip
!unzip -q ghidra_12.1.3_PUBLIC_20260817.zip && rm ghidra_12.1.3_PUBLIC_20260817.zip

# 2. Configure System Paths
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-25-openjdk-amd64"
os.environ["GHIDRA_INSTALL_DIR"] = "/content/ghidra_12.1.3_PUBLIC"

# 3. Install PyGhidra Python bindings
!pip install -q pyghidra ghidra-stubs==12.1.3


# 4. Install pyghidra and its core bridge dependencies
!pip install pyghidra -q

# Start PyGhidra

The `start()` function initialized the Ghidra backend. It must be called before any operation so that the engine is running inside the python interpreter.

In [ ]:
import pyghidra
pyghidra.start()

# The binaries

We have chosen small and simple, common Linux binaries that are present in the underlying operative system. This is a safe and simple way of running a mockup of a real-world analysis. If you were to do a malware analysis pipeline this part would point to a directory of binaries from a malware corpus.

In [ ]:
binaries = [
    "/bin/echo",
    "/bin/ls",
    "/bin/cat",
    "/bin/true",
    "/bin/false"
]
print(binaries)

# Feature Extraction with PyGhidra

For each binary we extract two numbers: the file size in kilobytes and the function count. The `program_context` API gives us access to Ghidra's anayzed program. While `listing.getFunctions(True)` iterates through all functions in the program. Function count is a simple static feature that correlates to code complexity. Real malware clasifiers use richer featers like API clals or n-gram frequencies but for now function count is good enough to show this pipeline.

In [ ]:
data = []

for binary in binaries:
  #Get the file size of each binary, the independent variable or input
  file_size = os.path.getsize(binary)

  # Changed to open_program to directly load the binary without requiring a project
  with pyghidra.open_program(binary) as flat_api:
    program = flat_api.getCurrentProgram()
    listing = program.getListing()
    func_count = sum(1 for _ in listing.getFunctions(True))

  data.append({"binary": binary, "size_kb": file_size/1024, "functions":func_count})

print(data)

# Create a Dataframe in Pandas

Now we use the data with three columns: binary path, size in KB and function count. Lets also see the raw data.

In [ ]:
import pandas as pd
df = pd.DataFrame(data)
df

# Visualization

It is important to **visualize the data before we model**. Normally we decide the model based on what we **know** about the data, or what we **see** from the behavior of the data. We should have some **domain knowledge** that allows us to create a hypothesis that gives us ideas to decide the model we will use. We can also use analogies from other subject areas to decide our model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

X = df["size_kb"].values
y = df["functions"].values

plt.figure(figsize=(8,5))
plt.scatter(X,y, color="steelblue",s=80,label="Observed binaries")
plt.xlabel("File Size (KB)")
plt.ylabel("Function Count")
plt.title("Binary Size vs. Function Count")
plt.legend()
plt.grid(True,alpha=0.3)
plt.show()

# Polynomial Features

We are using a quadratic model of the form $y = ax^2 + bx + c$. The `PolynomialFeatures` transforms our feature into $x$ and $x^2$. Then we can use a linear regression to fit the coefficients. There are other ways of doing this but we keep it simple for now.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X.reshape(-1,1))

# Fit the quadratic model

We have a three parameter model, these are $a$,$b$,$c$. These coefficients determine the shape of the curve. As we know from precalculus the sign of $a$ determines which way it opens.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_poly,y)
print("Coefficients:",model.coef_)
print("Intercept:",model.intercept_)

# Visalize the Quadratic Fit

The orange curve is the prediction of the model. You can observe from the plot if your fit agrees with the data. Since we only have 5 points our fit may not be perfect, you need significantly much more data for each additional parameter of your model. To create a complex model we need large quantities of data.

In [ ]:
X_plot = np.linspace(X.min(),X.max(),100).reshape(-1,1)
X_plot_poly = poly.transform(X_plot)
y_plot = model.predict(X_plot_poly)

plt.figure(figsize=(8,5))
plt.scatter(X, y, color="steelblue", s=80,label="Observed binaries")
plt.plot(X_plot, y_plot, color="darkorange",linewidth=2,label="Quadratic Fit")
plt.xlabel("File Size (KB)")
plt.ylabel("Function Count")
plt.title("Quadratic Regression: Binary Size vs. Function Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Make a Prediction

The idea to develop models is to **predict** an output an unknown input. *For a given value of file size, what would the expected function count be?* If your model describes the expected correlation between file size and function count for the files that you trust, **outliers** would catch your attention.

This is a simplified version of **anomaly detection** used in malware analysis.

In [ ]:
new_size = 50.0 #KB
new_poly = poly.transform([[new_size]])
predicted_functions = model.predict(new_poly)

print(f"Predicted function count for a {new_size} KB binary: {predicted_functions[0]:.1f}")

# Try it yourself

If we eliminate all the links in the folder that point out to other functions outside the folder, and restrict the file size to all files under 1 MB we can get significantly more data points.

In [ ]:
from pathlib import Path

directory_path = Path('/bin/')
## We chose 60 KB as the maximum. Increasing the vlaue will make it slower.
SIZE_LIMIT_BYTES = 0.06 * 1024 * 1024  # Change the leading number to a smaller number.

# Get files that are NOT symlinks and exceed the size limit
binaries = []
for f in directory_path.iterdir():
    if f.is_file() and not f.is_symlink() and f.stat().st_size < SIZE_LIMIT_BYTES:
        try:
            # Read the first 4 bytes to verify if it is a compiled ELF binary
            with open(f, "rb") as open_file:
                if open_file.read(4) == b"\x7fELF":
                    binaries.append(str(f))
        except OSError:
            # Skip files that can't be read (e.g., permission errors)
            continue
binaries = [item for item in binaries if "+" not in item]
print(binaries)
print(len(binaries))

# Process the larger file list with PyGhidra

In [ ]:
import os
data = []

for binary in binaries:
  #Get the file size of each binary, the independent variable or input
  file_size = os.path.getsize(binary)

  # Changed to open_program to directly load the binary without requiring a project
  with pyghidra.open_program(binary) as flat_api:
    program = flat_api.getCurrentProgram()
    listing = program.getListing()
    func_count = sum(1 for _ in listing.getFunctions(True))

  data.append({"binary": binary, "size_kb": file_size/1024, "functions":func_count})


# Create a Pandas Data Frame and Visualize the data

In [ ]:
df = pd.DataFrame(data)
X = df["size_kb"].values
y = df["functions"].values

plt.figure(figsize=(8,5))
plt.scatter(X,y, color="steelblue",s=80,label="Observed binaries")
plt.xlabel("File Size (KB)")
plt.ylabel("Function Count")
plt.title("Binary Size vs. Function Count")
plt.legend()
plt.grid(True,alpha=0.3)
plt.show()

# Model the Data

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X.reshape(-1,1))
model = LinearRegression()
model.fit(X_poly,y)
print("Coefficients:",model.coef_)
print("Intercept:",model.intercept_)

# Plot your model

In [ ]:
X_plot = np.linspace(X.min(),X.max(),100).reshape(-1,1)
X_plot_poly = poly.transform(X_plot)
y_plot = model.predict(X_plot_poly)

plt.figure(figsize=(8,5))
plt.scatter(X, y, color="steelblue", s=80,label="Observed binaries")
plt.plot(X_plot, y_plot, color="darkorange",linewidth=2,label="Quadratic Fit")
plt.xlabel("File Size (KB)")
plt.ylabel("Function Count")
plt.title("Quadratic Regression: Binary Size vs. Function Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()